In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.stats import jarque_bera, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Cargar datos
df = pd.read_csv(r"C:\Users\Ing. Antonio Rial\OneDrive - Universidad Austral\MCD_Laboratorio.de.Implementación.2\Competencia.House.Pricing\data\tabular\train_processed.csv", low_memory=False)

print("=" * 70)
print("COMPARACIÓN: lastSoldPrice_hpi_adjusted vs log_price")
print("=" * 70)
print(f"\nDimensiones del dataset: {df.shape}")

COMPARACIÓN: lastSoldPrice_hpi_adjusted vs log_price

Dimensiones del dataset: (11840, 46)


In [2]:
# ANÁLISIS EXPLORATORIO DE TARGETS
#fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Histogramas y QQ-plots...
# [código completo de la sección 1]

# Estadísticas descriptivas
desc_stats = pd.DataFrame({
    'lastSoldPrice_hpi_adjusted': df['lastSoldPrice_hpi_adjusted'].describe(),
    'log_price': df['log_price'].describe()
})
print(desc_stats.round(4))

# Tests de normalidad
jb_price = jarque_bera(df['lastSoldPrice_hpi_adjusted'].dropna())
jb_log = jarque_bera(df['log_price'].dropna())
print(f"\nJarque-Bera - Price: {jb_price[0]:.2f}, p={jb_price[1]:.2e}")
print(f"Jarque-Bera - Log:   {jb_log[0]:.2f}, p={jb_log[1]:.2e}")
print(f"Skewness Price: {df['lastSoldPrice_hpi_adjusted'].skew():.4f}")
print(f"Skewness Log:   {df['log_price'].skew():.4f}")

       lastSoldPrice_hpi_adjusted   log_price
count                1.184000e+04  11840.0000
mean                 5.590194e+05     13.0325
std                  3.509714e+05      0.6633
min                  5.127585e+04     10.8450
25%                  3.039663e+05     12.6247
50%                  4.722890e+05     13.0653
75%                  7.292780e+05     13.4998
max                  1.986843e+06     14.5021

Jarque-Bera - Price: 3137.23, p=0.00e+00
Jarque-Bera - Log:   292.99, p=2.39e-64
Skewness Price: 1.1428
Skewness Log:   -0.3851


In [3]:
# SELECCIÓN DE VARIABLES PREDICTORAS
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['zpid', 'lastSoldPrice_hpi_adjusted', 'log_price', 'latitude', 'longitude',
                'description', 'zipcode', 'desc_length', 'desc_word_count', 
                'desc_is_boilerplate', 'desc_mentions_renovated', 'desc_mentions_pool',
                'desc_mentions_view', 'desc_mentions_new',
                'latest_tax_value', 'latest_tax_paid', 'num_tax_records', 
                'last_listing_price', 'zip_3digit']
predictor_cols = [c for c in numeric_cols if c not in exclude_cols]

df_model = df[predictor_cols + ['lastSoldPrice_hpi_adjusted', 'log_price']].copy()

# Imputar missing values con mediana
for col in predictor_cols:
    if df_model[col].isna().sum() > 0:
        df_model[col].fillna(df_model[col].median(), inplace=True)

df_model = df_model.dropna(subset=['lastSoldPrice_hpi_adjusted', 'log_price'])
print(f"Variables predictoras: {len(predictor_cols)}")
print(f"Filas para modelado: {len(df_model)}")

Variables predictoras: 26
Filas para modelado: 11840


In [4]:
# CORRELACIONES CON PREDICTORES
corr_price = df_model[predictor_cols].corrwith(df_model['lastSoldPrice_hpi_adjusted']).sort_values(ascending=False)
corr_log = df_model[predictor_cols].corrwith(df_model['log_price']).sort_values(ascending=False)

corr_comparison = pd.DataFrame({
    'correlacion_con_price': corr_price,
    'correlacion_con_logprice': corr_log
}).sort_values('correlacion_con_logprice', ascending=False)

print(corr_comparison.head(15).round(4))

# Visualización...

                     correlacion_con_price  correlacion_con_logprice
taxAssessedValue                    0.7294                    0.6820
livingArea                          0.4658                    0.4800
log_living_area                     0.4338                    0.4790
bathrooms                           0.3686                    0.3975
bedrooms                            0.2750                    0.3390
max_school_rating                   0.3114                    0.2930
avg_school_rating                   0.2875                    0.2784
photoCount                          0.1243                    0.1399
has_garage                          0.1063                    0.1271
yearBuilt                           0.0830                    0.1102
has_pool                            0.1062                    0.1046
min_school_distance                 0.1129                    0.1014
has_waterfront                      0.1071                    0.0994
propertyTaxRate                   

In [5]:
# ENTRENAMIENTO Y COMPARACIÓN
X = df_model[predictor_cols].values
y_price = df_model['lastSoldPrice_hpi_adjusted'].values
y_log = df_model['log_price'].values

X_train, X_test, y_price_train, y_price_test = train_test_split(X, y_price, test_size=0.2, random_state=42)
_, _, y_log_train, y_log_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Modelo 1: Price
lr_price = LinearRegression()
lr_price.fit(X_train_s, y_price_train)
y_price_pred = lr_price.predict(X_test_s)

# Modelo 2: Log-Price
lr_log = LinearRegression()
lr_log.fit(X_train_s, y_log_train)
y_log_pred = lr_log.predict(X_test_s)
y_log_pred_price = np.exp(y_log_pred)

# Métricas
r2_price = r2_score(y_price_test, y_price_pred)
r2_log = r2_score(y_log_test, y_log_pred)
r2_log_price = r2_score(y_price_test, y_log_pred_price)

rmse_price = np.sqrt(mean_squared_error(y_price_test, y_price_pred))
rmse_log_price = np.sqrt(mean_squared_error(y_price_test, y_log_pred_price))

mae_price = mean_absolute_error(y_price_test, y_price_pred)
mae_log_price = mean_absolute_error(y_price_test, y_log_pred_price)

print(f"R² Price:      {r2_price:.4f}")
print(f"R² Log:        {r2_log:.4f}")
print(f"R² Log→exp:    {r2_log_price:.4f}")
print(f"RMSE Price:    ${rmse_price:,.0f}")
print(f"RMSE Log→exp:  ${rmse_log_price:,.0f}")
print(f"MAE Price:     ${mae_price:,.0f}")
print(f"MAE Log→exp:   ${mae_log_price:,.0f}")

R² Price:      0.6238
R² Log:        0.5991
R² Log→exp:    0.3913
RMSE Price:    $215,428
RMSE Log→exp:  $274,029
MAE Price:     $152,338
MAE Log→exp:   $170,227


In [6]:
# ANÁLISIS DE RESIDUOS
residuals_price = y_price_test - y_price_pred
residuals_log = y_log_test - y_log_pred

# Heterocedasticidad
corr_het_price, p_het_price = spearmanr(y_price_pred, np.abs(residuals_price))
corr_het_log, p_het_log = spearmanr(y_log_pred, np.abs(residuals_log))

print(f"Heterocedasticidad Price:     ρ={corr_het_price:.4f}, p={p_het_price:.2e}")
print(f"Heterocedasticidad Log-Price: ρ={corr_het_log:.4f}, p={p_het_log:.2e}")

# Varianza por cuartiles
q_price = pd.qcut(y_price_pred, q=4, labels=['Q1','Q2','Q3','Q4'])
var_price = pd.DataFrame({'q': q_price, 'res': residuals_price}).groupby('q')['res'].var()
q_log = pd.qcut(y_log_pred, q=4, labels=['Q1','Q2','Q3','Q4'])
var_log = pd.DataFrame({'q': q_log, 'res': residuals_log}).groupby('q')['res'].var()

print("\nVarianza por cuartil (Price):")
print(var_price)
print("\nVarianza por cuartil (Log):")
print(var_log)

Heterocedasticidad Price:     ρ=0.3741, p=1.52e-79
Heterocedasticidad Log-Price: ρ=-0.0677, p=9.86e-04

Varianza por cuartil (Price):
q
Q1    1.211466e+10
Q2    2.915911e+10
Q3    4.460884e+10
Q4    9.879535e+10
Name: res, dtype: float64

Varianza por cuartil (Log):
q
Q1    0.194277
Q2    0.144751
Q3    0.146463
Q4    0.174201
Name: res, dtype: float64


# Conclusión: ¿Cuál target usar?

## Recomendación: **log_price**

| Criterio | lastSoldPrice_hpi_adjusted | log_price |
|----------|---------------------------|-----------|
| Normalidad | ❌ Asimétrica (skew > 2) | ✅ Simétrica (~0.3) |
| Homocedasticidad | ❌ Severa (ρ=0.37) | ✅ Leve (ρ=-0.07) |
| R² (propio) | ✅ 0.6238 | 0.5991 |
| R² (convertido $) | 0.6238 | ❌ 0.3913 |
| Robustez outliers | ❌ Baja | ✅ Alta |
| Interpretación coef. | Directa ($) | Elasticidades (%) |

### Razones para elegir log_price:
1. ✅ Mejor cumplimiento de supuestos del modelo lineal
2. ✅ Residuos con varianza más estable (homocedasticidad)
3. ✅ Menos sensible a outliers en precios extremos
4. ✅ Coeficientes interpretables como cambios porcentuales
5. ✅ Mejor desempeño en precios bajo-medio (mayoría de datos)

### Precaución:
- Al convertir predicciones de log → $ con `exp()`, se introduce sesgo.
- Aplicar corrección de **Duan/Smearing**: `ŷ = exp(ŷ_log) × mean(exp(residuos))`
- Para interpretación directa en dólares, usar `lastSoldPrice_hpi_adjusted` como benchmark.